# Databricks connection smoke test

Run the cells top to bottom. It loads `.env`, sanity-checks the three connection vars (without printing the token), opens a SQL connection, runs `SELECT 1`, and lists the `lakehouse-workshop` catalog.

Use the project venv as the kernel (`.venv`). If `databricks-sql-connector` is missing, run `%pip install databricks-sql-connector python-dotenv` in a cell.

In [7]:
# 1) Load .env and sanity-check the vars (does NOT print the token)
import os
from pathlib import Path
from dotenv import load_dotenv

# find .env from the notebook location, falling back to the repo root
candidates = [Path.cwd() / '.env', Path.cwd().parent / '.env', Path('/Users/zach/dev/genai-workshop-lakehouse/.env')]
env_path = next((p for p in candidates if p.exists()), None)
print('loading .env from:', env_path)
load_dotenv(env_path, override=True)

host = os.environ.get('DATABRICKS_SERVER_HOSTNAME', '')
path = os.environ.get('DATABRICKS_HTTP_PATH', '')
tok  = os.environ.get('DATABRICKS_TOKEN', '')

print('HOST        :', repr(host))                      # safe to show
print('HTTP_PATH   :', repr(path))                      # safe to show
print('TOKEN set?  :', bool(tok), '| length:', len(tok), '| starts dapi:', tok.startswith('dapi'))

# common gotchas
if host.startswith('http'):       print('  !! HOST should NOT include https:// — strip it')
if not path.startswith('/sql/'):  print('  !! HTTP_PATH should look like /sql/1.0/warehouses/<id> (a SQL warehouse)')
if tok.strip() != tok:            print('  !! TOKEN has leading/trailing whitespace')
if tok.startswith((chr(34), chr(39))): print('  !! TOKEN looks quoted — remove the surrounding quotes')
if not tok:                       print('  !! TOKEN is empty — nothing will authenticate')

loading .env from: /Users/zach/dev/genai-workshop-lakehouse/.env
HOST        : 'dbc-cc887abc-9779.cloud.databricks.com'
HTTP_PATH   : '/sql/1.0/warehouses/1aaaa44393934a02'
TOKEN set?  : True | length: 36 | starts dapi: True


In [9]:
# 2) Open a connection and run SELECT 1
from databricks import sql

with sql.connect(server_hostname=host, http_path=path, access_token=tok) as conn:
    with conn.cursor() as cur:
        cur.execute('SELECT 1 AS ok, current_user() AS who, current_version().dbsql_version AS dbsql')
        print('CONNECTED:', cur.fetchone())

CONNECTED: Row(ok=1, who='zach.blumenfeld@neo4j.com', dbsql='2026.15')


In [10]:
# 3) Inventory the lakehouse-workshop catalog (so we can see what to nuke)
CAT = 'lakehouse-workshop'
with sql.connect(server_hostname=host, http_path=path, access_token=tok) as conn:
    with conn.cursor() as cur:
        cur.execute(f'SHOW SCHEMAS IN `{CAT}`')
        schemas = [r[0] for r in cur.fetchall()]
        print(f'`{CAT}` schemas:', schemas)
        for s in schemas:
            if s == 'information_schema':
                continue
            cur.execute(f'SHOW TABLES IN `{CAT}`.`{s}`')
            tbls = [r[1] for r in cur.fetchall()]
            print(f'  `{s}`:', tbls)

`lakehouse-workshop` schemas: ['autofix_service', 'default', 'information_schema']
  `autofix_service`: ['dtc_codes', 'parts', 'procedures', 'vehicles', 'work_order_parts', 'work_orders']
  `default`: []
